In [3]:
import numpy as np
import pandas as pd
from hmmlearn import hmm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

In [4]:
file = "../data/dataset1_3x40_clean.csv"
df = pd.read_csv(file)

In [5]:
signals = ["ax", "ay", "az", "gx", "gy", "gz", "alpha", "beta", "gamma"]
T = 60

In [6]:
def row_to_timeseries(row, signals, T):
    """
    Transforme une ligne du dataframe en série temporelle (T, D)
    """
    sequence = []

    for t in range(1, T + 1):
        features_t = [row[f"{sig}_{t}"] for sig in signals]
        sequence.append(features_t)

    return np.array(sequence)  # shape (T, 9)

In [7]:
data_by_move = defaultdict(list)

for _, row in df.iterrows():
    label = row["label"]
    seq = row_to_timeseries(row, signals, T)
    data_by_move[label].append(seq)

In [8]:
X_train = {}
X_test = {}

for move, sequences in data_by_move.items():
    sequences = np.array(sequences)

    assert len(sequences) >= 40, f"{move} n'a pas assez d'exemples"

    X_train[move] = sequences[:30]
    X_test[move] = sequences[30:40]

In [9]:
scaler = StandardScaler()

train_all = np.vstack([
    seq.reshape(-1, seq.shape[-1])
    for move in X_train
    for seq in X_train[move]
])

scaler.fit(train_all)

def normalize_sequences(seqs, scaler):
    return [scaler.transform(seq) for seq in seqs]

for move in X_train:
    X_train[move] = normalize_sequences(X_train[move], scaler)
    X_test[move] = normalize_sequences(X_test[move], scaler)

In [10]:
models = {}
n_states = 6  # hyperparamètre à ajuster

for move, sequences in X_train.items():
    model = hmm.GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=300,
        random_state=42
    )

    X_concat = np.vstack(sequences)
    lengths = [len(seq) for seq in sequences]

    model.fit(X_concat, lengths)

    models[move] = model
    print(f"HMM entraîné pour {move}")

HMM entraîné pour move_1
HMM entraîné pour move_2
HMM entraîné pour move_3


In [11]:
def predict(sequence, models):
    scores = {}
    for move, model in models.items():
        scores[move] = model.score(sequence)
    return max(scores, key=scores.get), scores

In [12]:
correct = 0
total = 0

for true_move, sequences in X_test.items():
    for seq in sequences:
        pred, scores = predict(seq, models)
        total += 1
        correct += (pred == true_move)

        print(f"Vrai: {true_move} | Prédit: {pred}")

print(f"\nAccuracy: {correct / total:.2%}")

Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_1 | Prédit: move_1
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_2 | Prédit: move_2
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3
Vrai: move_3 | Prédit: move_3

Accuracy: 100.00%


In [13]:
import joblib
from pathlib import Path

out_dir = Path("models")
out_dir.mkdir(exist_ok=True)

joblib.dump(scaler, out_dir / "scaler.joblib")
joblib.dump(models, out_dir / "hmm_models.joblib")

print("✅ Saved:", out_dir / "scaler.joblib", "and", out_dir / "hmm_models.joblib")

✅ Saved: models/scaler.joblib and models/hmm_models.joblib
